In [ ]:
import pandas as pd
from common.utils import get_project_root
from typing import Iterable
from packages.custom_nlp.src.custom_nlp.tratamento_texto import renomeia_cols

In [ ]:
texto_para_remover = "VERSÃO PRELIMINAR (Esta versão será substituída após conclusão da revisão de perfis, conhecimentos, habilidades, atitudes e níveis)"


def agg_f(textos: Iterable[str]) -> str:
    textos_sem_versao_preliminar = set(
        map(lambda texto: texto.replace(texto_para_remover, "").strip(), textos)
    )
    return ". ".join(textos_sem_versao_preliminar)


def agg_textos(
    df: pd.DataFrame, agg_col: str, cols_with_text: list[str]
) -> pd.DataFrame:
    return (
        df.dropna(subset=cols_with_text)
        .groupby(by=agg_col, as_index=False)[cols_with_text]
        .agg(agg_f)
        .rename(columns={agg_col: "codigo"})
    )

In [28]:
project_root_dir = get_project_root("classificador-cbo")

# Carregando os dados
qbq_path = project_root_dir / "data/bronze/OcupacoesCBO.xlsx"
cols_interesse = ["CodCBO", "Ocupação", "Síntese", "PerfilOcupacional"]
df = pd.read_excel(qbq_path, sheet_name="Ocupação", usecols=cols_interesse, dtype=str)

# Pegando os códigos
df["grande_grupo"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:1])
df["subgrupo_principal"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:2])
df["subgrupo"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:3])
df["familia"] = df.loc[:, "CodCBO"].apply(lambda codigo: codigo[:4])

#
classificacoes = [
    ("grande_grupo", "Grande Grupo"),
    ("subgrupo_principal", "SubGrupo Principal"),
    ("subgrupo", "SubGrupo"),
    ("familia", "Familia"),
]

for classificacao in classificacoes:
    # Obtendo os textos
    textos = renomeia_cols(
        agg_textos(df, classificacao[0], ["Síntese", "PerfilOcupacional"])
    )
    path_csv_titulos = (
        project_root_dir / f"data/bronze/CBO2002 - {classificacao[1]}.csv"
    )

    titulos = renomeia_cols(
        pd.read_csv(path_csv_titulos, dtype=str, encoding="latin1", sep=";")
    )

    textos_e_titulos = pd.merge(left=textos, right=titulos, on="codigo", how="left")

    path_csv_destino = project_root_dir / f"data/silver/{classificacao[0]}.csv"
    textos_e_titulos.to_csv(path_csv_destino, index=False, sep="\t", encoding="utf-8")

# A

In [5]:
# =============================================================================
# BIBLIOTECAS E MÓDULOS
# =============================================================================

import numpy as np
import pandas as pd
from common.utils import get_project_root
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from custom_nlp.tratamento_texto import Preprocessor, NormalizationStrategy, StopwordsRemovalStrategy
from custom_nlp.embeddings import Embedder, CountVect, TfidfVect

# =============================================================================
# CONSTANTES
# =============================================================================

project_root_dir = get_project_root("classificador-cbo")
treated_data_dir = project_root_dir / f"data/silver/cbo_sintese_perfil"
teste = """**Título:** Contador Financeiro  

**Descrição:**  
Estamos em busca de um contador financeiro altamente qualificado para integrar nossa equipe. O profissional será responsável por gerenciar e analisar as demonstrações financeiras, garantindo a conformidade com as normas contábeis e fiscais.  

**Responsabilidades:**  
- Elaborar e analisar relatórios financeiros detalhados  
- Monitorar e garantir a conformidade fiscal e contábil  
- Gerenciar fluxo de caixa e planejamento financeiro  
- Fornecer suporte contábil para decisões estratégicas  
- Trabalhar em conjunto com outras áreas para garantir a precisão das informações financeiras  

**Requisitos:**  
- Graduação em Contabilidade, Finanças ou área relacionada  
- Experiência prévia na área contábil ou financeira  
- Conhecimento em normas contábeis e legislação fiscal  
- Domínio de ferramentas financeiras e contábeis  
- Habilidades analíticas e atenção aos detalhes"""

# =============================================================================
# CONSTANTES
# =============================================================================

path_csv = treated_data_dir / "familia.csv"
df = pd.read_csv(path_csv, sep="\t").dropna()

textos = df.loc[:,"sintese"].values

#
preprocessador = Preprocessor(strategy=NormalizationStrategy)
textos_normalizados = np.vectorize(pyfunc=preprocessador.apply)(textos)

preprocessador.set_strategy(strategy=StopwordsRemovalStrategy)
textos_normalizados_s_stopwords = np.vectorize(pyfunc=preprocessador.apply)(textos_normalizados)

#
count_vect = CountVectorizer()
count_vect.fit(raw_documents=textos_normalizados_s_stopwords)

#
tfidf_vect = TfidfVectorizer()
tfidf_vect.fit(raw_documents=textos_normalizados_s_stopwords)

#
embedder = Embedder(strategy=CountVect, model=count_vect)
embedding_ref_count = embedder.apply(textos_normalizados_s_stopwords)
embedding_texto_count = embedder.apply(teste)

embedder.set_strategy(strategy=TfidfVect, model=tfidf_vect)
embedding_ref_tfidf = embedder.apply(textos_normalizados_s_stopwords)
embedding_texto_tfidf = embedder.apply(teste)

In [13]:
import numpy as np

dot_count = np.dot(embedding_ref_count, embedding_texto_count.T)
idx_argmax_count = dot_count.argmax()
print(df.iloc[idx_argmax_count,-1])

dot_tfidf = np.dot(embedding_ref_tfidf, embedding_texto_tfidf.T)
idx_argmax_tfidf = dot_tfidf.argmax()
print(df.iloc[idx_argmax_tfidf,-1])
# a[idx_argmax]

Médicos clínicos
Contadores e afins


In [15]:
df.iloc[116]

codigo                                                            2522
sintese              Realiza perícias contábeis judiciais, extrajud...
perfilocupacional    Realiza perícias contábeis judiciais, extrajud...
titulo                                              Contadores e afins
Name: 116, dtype: object

In [4]:
for a in treated_data_dir.rglob("*.csv"):
    print(a)

/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/grande_grupo.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo_principal.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/subgrupo.csv
/Users/alecrim/workspace/classificador-cbo/data/silver/cbo_sintese_perfil/familia.csv
